# Lyrics ➜ Audio ➜ Anki

Generates sentence audio for `sentence-zh` batches with **CosyVoice 2** (Apache-2.0, Chinese-first) on a free GPU, then builds the `.apkg` into `MyDrive/Anki/`.

**Before running:** *Runtime ▸ Change runtime type ▸ T4 GPU*.

Everything happens in one session, so the audio doesn't have to be committed to the repo — but cell 6 lets you download it if you want to keep it there.

In [ ]:
#@title 1. Setup
%pip -q install genanki pypinyin
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

In [ ]:
#@title 2. Get the latest repo
import os
if os.path.isdir('/content/Anki'):
    !git -C /content/Anki pull
else:
    !git clone --depth 1 https://github.com/Nezv/Anki /content/Anki

In [ ]:
#@title 3. Install CosyVoice 2 (~5 min, first run only)
import os, sys
if not os.path.isdir('/content/CosyVoice'):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice /content/CosyVoice
%pip -q install -r /content/CosyVoice/requirements.txt
sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

from modelscope import snapshot_download
snapshot_download('iic/CosyVoice2-0.5B', local_dir='/content/pretrained_models/CosyVoice2-0.5B')

### 4. Reference voice

CosyVoice 2 clones the voice from a short reference clip. Upload a **3–10 s** clean mono wav of the voice you want (a calm native speaker reading a sentence works best — not the song itself), and type exactly what is said in it.

In [ ]:
#@title 4. Upload the reference clip
from google.colab import files
uploaded = files.upload()
REF_AUDIO = '/content/' + next(iter(uploaded))
REF_TEXT  = '\u6211\u5f88\u9ad8\u5174\u8ba4\u8bc6\u4f60\u3002'  #@param {type:"string"}
print(REF_AUDIO, REF_TEXT)

In [ ]:
#@title 5. Generate the audio
BATCH = '/content/Anki/batches/zh-*.txt'  #@param {type:"string"}

import os
os.environ['PYTHONPATH'] = '/content/CosyVoice:/content/CosyVoice/third_party/Matcha-TTS'
!python /content/Anki/tools/gen_audio.py {BATCH} \
    --engine cosyvoice \
    --model-dir /content/pretrained_models/CosyVoice2-0.5B \
    --ref-audio "{REF_AUDIO}" --ref-text "{REF_TEXT}" \
    --media /content/Anki/media

In [ ]:
#@title 6. Build the .apkg into Drive
!mkdir -p '/content/drive/MyDrive/Anki'
!python /content/Anki/tools/build_apkg.py --out '/content/drive/MyDrive/Anki'

In [ ]:
#@title 7. (Optional) Keep the audio — copy media + patched batches to Drive
!mkdir -p '/content/drive/MyDrive/Anki/media'
!cp -r /content/Anki/media/. '/content/drive/MyDrive/Anki/media/'
!cp /content/Anki/batches/zh-*.txt '/content/drive/MyDrive/Anki/'
print('Commit these back to the repo if you want reproducible rebuilds.')